In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

sns.set_theme(color_codes=True,
              style='darkgrid')

In [ ]:
df = pd.read_csv("/kaggle/input/datasets/prasad22/retail-transactions-dataset/Retail_Transactions_Dataset.csv")

df.head()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.describe(include='O')

In [ ]:
df['Discount_Applied'].unique()

In [ ]:
col_name = df.columns
col_name

In [ ]:
df.nunique()

In [ ]:
df.drop(columns='Transaction_ID', inplace=True)

In [ ]:
df['Date'] = pd.to_datetime(df['Date'])

df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Day'] = df['Date'].dt.day
df.drop(columns='Date', inplace=True)

In [ ]:
plot_df = df.sample(10000, random_state=42)   

In [ ]:
sns.pairplot(data=plot_df, hue='Discount_Applied', palette='dark')
#sns.pairplot(df.sample(1000), hue='Discount_Applied', palette ='dark')

In [ ]:
fig = px.histogram(plot_df,
                   x='Discount_Applied',
                   color='Discount_Applied',
                   text_auto=True)

fig.update_layout(
    title='Distribution of Transactions Based on Discount',
    title_font={'size':30},
    showlegend=True
)

fig.update_yaxes(showgrid=False)
fig.show()

In [ ]:
df.groupby('Discount_Applied')['Payment_Method'].value_counts()

In [ ]:
sns.countplot(x='Payment_Method',
              hue='Discount_Applied',
              data=plot_df)

In [ ]:
plot_df.groupby('Discount_Applied')['Store_Type'].value_counts().plot.pie(
    autopct='%1.1f%%',
    figsize=(15,6)
)

plt.title('Discount vs Store Type')
plt.axis('equal')
plt.show()

In [ ]:
df.columns

In [ ]:
cat_cols = df.select_dtypes(include='object').columns
num_cols = df.select_dtypes(exclude='object').columns

num_cols, cat_cols

In [ ]:
fig, axs = plt.subplots(nrows=1, ncols=3, figsize=(20,10))
axs = axs.flatten()

top_n = 15  
for i, var in enumerate(cat_cols[:3]):
    top_categories = plot_df[var].value_counts().nlargest(top_n).index
    sns.countplot(
        x=var,
        hue='Discount_Applied',
        data=plot_df[plot_df[var].isin(top_categories)],
        ax=axs[i]
    )
    axs[i].set_xticklabels(axs[i].get_xticklabels(), rotation=45)
    axs[i].set_xlabel(var)

plt.tight_layout()
plt.show()

In [ ]:
 

top_products = plot_df['Product'].value_counts().nlargest(top_n).index

sns.histplot(
    data=plot_df[plot_df['Product'].isin(top_products)],
    x='Product',
    hue='Discount_Applied',
    multiple='fill',
    kde=False,
    element='bars',
    fill=True
)
plt.xticks(rotation=45)
plt.title(f'Distribution of Top {top_n} Products by Discount Applied')
plt.show()

In [ ]:
fig, axs = plt.subplots(nrows=1, ncols=3, figsize=(20,6))
axs = axs.flatten()

for i, var in enumerate(cat_cols[:3]):
    # get top N categories by frequency
    top_categories = plot_df[var].value_counts().nlargest(top_n).index
    
    sns.histplot(
        data=plot_df[plot_df[var].isin(top_categories)],
        x=var,
        hue='Discount_Applied',
        multiple='fill',
        kde=False,
        element='bars',
        ax=axs[i],
        fill=True
    )
    axs[i].set_xticklabels(axs[i].get_xticklabels(), rotation=45)
    axs[i].set_xlabel(var)

plt.tight_layout()
plt.show()

In [ ]:
num_cols

In [ ]:
fig, axs = plt.subplots(nrows=3, ncols=3, figsize=(20,10))
axs = axs.flatten()

for i, var in enumerate(num_cols):
    sns.boxplot(x=var, data=plot_df, ax=axs[i])

plt.tight_layout()
plt.show()

In [ ]:
fig, axs = plt.subplots(nrows=3, ncols=3, figsize=(20,20))
axs = axs.flatten()

for i, var in enumerate(num_cols):
    sns.boxplot(y=var,
                x='Discount_Applied',
                hue='Discount_Applied',
                data=plot_df,
                ax=axs[i])

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10,5))
sns.countplot(
    x='Total_Items',
    hue='Discount_Applied',
    data=plot_df
)
plt.title('Discount Applied vs Total Items Purchased')
plt.show()

In [ ]:
plt.figure(figsize=(10,5))
sns.countplot(
    x='Season',
    hue='Discount_Applied',
    data=plot_df
)
plt.title('Discount Applied vs Season')
plt.show()

In [ ]:
pivot = df.pivot_table(
    index='Customer_Category',
    columns='Discount_Applied',
    aggfunc={'Discount_Applied':'count'}
)

pivot.plot.pie(
    subplots=True,
    figsize=(20,10),
    autopct='%1.1f%%'
)

plt.title('Discount Applied Distribution Across Customer Categories')
plt.axis('equal')
plt.show()

In [ ]:
df.isnull().sum()

In [ ]:
for col in cat_cols:
    print(f"{col}: {df[col].unique()}")

In [ ]:
from sklearn.preprocessing import LabelEncoder

for col in cat_cols:
    le = LabelEncoder()
    le.fit(df[col].unique())
    df[col] = le.transform(df[col])

    print(f"{col}: {df[col].unique()}")

In [ ]:
corr = df.corr()

In [ ]:
plt.figure(figsize=(20,10))
sns.heatmap(corr, fmt='.2f', annot=True)
plt.title('Features affecting Discount Applied')
plt.show()

In [ ]:
max_5_corr = corr.nlargest(5, 'Discount_Applied')

plt.figure(figsize=(15,5))
sns.heatmap(max_5_corr, fmt='.2f', annot=True)
plt.show()

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop('Discount_Applied', axis=1)
y = df['Discount_Applied']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

In [ ]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression()
lr.fit(X_train, y_train)

In [ ]:
lr.score(X_train, y_train)


In [ ]:
lr.score(X_test, y_test)


In [ ]:
lr_pred = lr.predict(X_test)


In [ ]:

from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, jaccard_score, log_loss

accuracy_score(y_test, lr_pred)
     

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, jaccard_score

def model_accuracy(model, model_name):

    y_pred = model.predict(X_test)

    print(f"---------{model_name}---------")
    print(f'Accuracy : {round(accuracy_score(y_test,y_pred)*100,2)}%')
    print(f"F1 Score: {round(f1_score(y_test,y_pred,average='micro'),2)}")
    print(f"Precision Score: {round(precision_score(y_test,y_pred,average='micro'),2)}")
    print(f"Recall Score: {round(recall_score(y_test,y_pred,average='micro'),2)}")
    print(f"Jaccard Score: {round(jaccard_score(y_test,y_pred,average='micro'),2)}")

In [ ]:
model_accuracy(lr, "Logistic Regression")

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV

dtree = DecisionTreeClassifier()
dtree.fit(X_train,y_train)

model_accuracy(dtree,"DecisionTree before GRID SEARCH")

In [ ]:

param_grid = {
    'max_depth':[3,4,5,6,7,8],
    'min_samples_split':[2,3,4],
    'min_samples_leaf':[1,2,3,4],
    'random_state':[0,42]
}

grid_search = GridSearchCV(dtree,param_grid,cv=5)
#grid_search.fit(X_train,y_train)
X_sample = X_train.sample(20000, random_state=42)
y_sample = y_train.loc[X_sample.index]

grid_search.fit(X_sample, y_sample)

In [ ]:
grid_search.best_params_


In [ ]:
dtree = DecisionTreeClassifier(
    random_state=42,
    max_depth=5,
    min_samples_split=2,
    min_samples_leaf=2
)

dtree.fit(X_train, y_train)
y_pred_tree = dtree.predict(X_test)
model_accuracy(dtree, "Decision Tree After GRID SEARCH")

In [ ]:
imp_df = pd.DataFrame({
    'Feature name': X_train.columns,
    'importance': dtree.feature_importances_
}).sort_values(by='importance',ascending=False).head(10)

sns.barplot(data=imp_df,x='importance',y='Feature name')
plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, dtree.predict(X_test))
sns.heatmap(cm, annot=True, cmap="Blues")
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

In [ ]:
from sklearn.ensemble import RandomForestClassifier

X_sample = X_train.sample(20000, random_state=42)
y_sample = y_train.loc[X_sample.index]

rforest = RandomForestClassifier(
    n_estimators=20,   # few trees for speed
    max_depth=8,       # limit depth
    n_jobs=-1,
    random_state=42
)

rforest.fit(X_sample, y_sample)
model_accuracy(rforest, "RandomForest (subset, fast)")

In [ ]:
param_grid={
    'n_estimators':[50,100],   # fewer trees for speed
    'max_depth':[5,8],         # limit depth
    'max_features':['sqrt'],
    'random_state':[0,42]
}


grid_searchrf = GridSearchCV(
    rforest,
    param_grid,
    cv=5,
    n_jobs=-1,
    verbose=2
)

#grid_searchrf = GridSearchCV(rforest,param_grid,cv=5)
#grid_searchrf.fit(X_train,y_train)
X_sample = X_train.sample(20000, random_state=42)
y_sample = y_train.loc[X_sample.index]

grid_searchrf.fit(X_sample, y_sample)

In [ ]:
grid_searchrf.best_params_


In [ ]:
X_sample = X_train.sample(20000, random_state=42)
y_sample = y_train.loc[X_sample.index]
rforest2 = RandomForestClassifier(**grid_searchrf.best_params_)
rforest2.fit(X_sample,y_sample)

model_accuracy(rforest2,"RandomForest after CV")

In [ ]:
def makeConfusionMatrix(model, modelName):
    y_pred = model.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)

    sns.heatmap(cm, annot=True, cmap="Blues")
    plt.title(f"Confusion Matrix for {modelName}")
    plt.show()

In [ ]:
makeConfusionMatrix(dtree,"Decision Tree")
makeConfusionMatrix(lr,"Logistic Regression")
makeConfusionMatrix(rforest2,"Random Forest")

In [ ]:
model_accuracy(dtree, 'Decision Tree')
model_accuracy(lr, "Logisitic Regression")
model_accuracy(rforest2,"RandomForest")
     

In [ ]:
import joblib

joblib.dump(dtree,'DecisionTree_Retail.pkl')